# We now have data for cases and vaccine coverage for these three diseases
- measles
- mumps
- pertussis

Files
- app/data/tycho_cases.csv
- app/data/tycho_measles_control.csv

In [8]:
import pandas as pd

cases_df = pd.read_csv('../app/data/tycho_cases.csv')
cases_df.head()

,year,state,measles_cases,mumps_cases,pertussis_cases
0,1995,AK,NaN,13.0,1.0
1,1995,AL,NaN,4.0,38.0
2,1995,AR,2.0,10.0,41.0
3,1995,AZ,10.0,2.0,151.0
4,1995,CA,108.0,206.0,463.0


In [9]:
coverage_df = pd.read_csv('../app/data/nis_vacc_coverage.csv')
coverage_df.head()

,state,year,measles_coverage_pct,measles_n,measles_n_vaccinated,mumps_coverage_pct,mumps_n,mumps_n_vaccinated,pertussis_coverage_pct,pertussis_n,pertussis_n_vaccinated
0,AK,1995,89.86,194.0,177.0,89.86,194.0,177.0,77.58,194.0,159.0
1,AK,1996,84.55,260.0,225.0,84.55,260.0,225.0,78.25,260.0,210.0
2,AK,1997,87.41,291.0,257.0,87.41,291.0,257.0,80.98,291.0,242.0
3,AK,1998,87.06,34.0,30.0,87.06,34.0,30.0,82.01,34.0,28.0
4,AK,1999,90.67,349.0,321.0,90.67,349.0,321.0,83.54,349.0,296.0


In [10]:
yearly_cases_df = cases_df.groupby('year')[['measles_cases', 'mumps_cases', 'pertussis_cases']].sum(min_count=1).reset_index()
yearly_cases_df.head()

,year,measles_cases,mumps_cases,pertussis_cases
0,1995,285.0,795.0,4048.0
1,1996,477.0,613.0,5930.0
2,1997,171.0,593.0,5247.0
3,1998,92.0,463.0,5811.0
4,1999,140.0,343.0,5123.0


In [11]:
yearly_coverage_df = coverage_df.groupby('year')[['measles_coverage_pct', 'mumps_coverage_pct', 'pertussis_coverage_pct']].mean().reset_index()
yearly_coverage_df.head()

,year,measles_coverage_pct,mumps_coverage_pct,pertussis_coverage_pct
0,1995,89.875882,89.861373,79.341961
1,1996,90.327647,90.279608,81.645490
2,1997,90.728431,90.602157,82.678235
3,1998,92.482549,92.341373,84.261961
4,1999,91.712353,91.555882,84.468235


In [12]:
yearly_coverage_cases_df = pd.merge(yearly_cases_df, yearly_coverage_df, on='year')
yearly_coverage_cases_df.head()

,year,measles_cases,mumps_cases,pertussis_cases,measles_coverage_pct,mumps_coverage_pct,pertussis_coverage_pct
0,1995,285.0,795.0,4048.0,89.875882,89.861373,79.341961
1,1996,477.0,613.0,5930.0,90.327647,90.279608,81.645490
2,1997,171.0,593.0,5247.0,90.728431,90.602157,82.678235
3,1998,92.0,463.0,5811.0,92.482549,92.341373,84.261961
4,1999,140.0,343.0,5123.0,91.712353,91.555882,84.468235


In [13]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

diseases = ["measles", "mumps", "pertussis"]

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    specs=[
        [{"secondary_y": True}],
        [{"secondary_y": True}],
        [{"secondary_y": True}]
    ],
    subplot_titles=["Measles", "Mumps", "Pertussis"],
    vertical_spacing=0.10
)

colors = {
    "measles": "#E74C3C",
    "mumps": "#3498DB",
    "pertussis": "#2ECC71"
}

for row, disease in enumerate(diseases, start=1):

    # Annual cases
    fig.add_trace(
        go.Bar(
            x=yearly_coverage_cases_df["year"],
            y=yearly_coverage_cases_df[f"{disease}_cases"],
            name=f"{disease.title()} cases",
            marker_color=colors[disease],
            opacity=0.65,
            hovertemplate=(
                "Year: %{x}<br>"
                "Cases: %{y:,.0f}"
                "<extra></extra>"
            )
        ),
        row=row,
        col=1,
        secondary_y=False
    )

    # Annual vaccination coverage
    fig.add_trace(
        go.Scatter(
            x=yearly_coverage_cases_df["year"],
            y=yearly_coverage_cases_df[f"{disease}_coverage_pct"],
            name=f"{disease.title()} coverage",
            mode="lines+markers",
            line=dict(color="black", width=2),
            marker=dict(size=6),
            hovertemplate=(
                "Year: %{x}<br>"
                "Coverage: %{y:.2f}%"
                "<extra></extra>"
            )
        ),
        row=row,
        col=1,
        secondary_y=True
    )

    fig.update_yaxes(
        title_text="Cases",
        row=row,
        col=1,
        secondary_y=False
    )

    fig.update_yaxes(
        title_text="Coverage (%)",
        row=row,
        col=1,
        secondary_y=True
    )

fig.update_xaxes(
    title_text="Year",
    row=3,
    col=1,
    dtick=1
)

fig.update_layout(
    title="Annual Vaccination Coverage and Reported Cases",
    template="plotly_white",
    height=900,
    hovermode="x unified",
    legend_title="Measure",
    bargap=0.20
)

fig.show()

## As you can see above, this serves as a more descriptive analysis as we are looking at a sum & average over the entire country

### Next, we should analyze be state but which ones?